In [ ]:
pip install langchain langchain_groq langchain_community python-dotenv

In [ ]:
from google.colab import userdata

# Retrieve the Groq API key from Colab secrets
GROQ_API_KEY = ""

# Ensure the .env file exists and write the key to it
with open('.env', 'a') as f:
    f.write(f'GROQ_API_KEY="{GROQ_API_KEY}"\n')

print("GROQ_API_KEY written to .env file.")

import os, json
from dotenv import load_dotenv
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq


load_dotenv()
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY
)

# --- shared state schema ---
class AgentState(TypedDict):
    goal:        str
    tasks:       List[str]
    results:     List[str]
    critique:    str
    approved:    bool
    iterations:  int

def planner(state: AgentState) -> AgentState:
    system = """You are a planning agent. Break the user's goal into
at most 5 concrete, actionable tasks. Respond ONLY with a
valid JSON array of strings. No preamble, no markdown."""

    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Goal: {state['goal']}")
    ]
    response = llm.invoke(messages).content.strip()

    try:
        clean = response.replace("json","\
").replace("","\
").strip()
        tasks = json.loads(clean)
    except json.JSONDecodeError:
        tasks = [response]   # fallback: treat whole response as one task

    print(f"\n[Planner] Generated {len(tasks)} tasks:")
    for i, t in enumerate(tasks): print(f"  {i+1}. {t}")

    return {**state, "tasks": tasks}

initial_state: AgentState = {
    "goal":        "Research and summarise the top 3 trends in agriculture for 2025",
    "tasks":       [],
    "results":     [],
    "critique":    "",
    "approved":    False,
    "iterations":  0
}
planner(initial_state)

GROQ_API_KEY written to .env file.

[Planner] Generated 5 tasks:
  1. Identify credible sources of agricultural research and trends
  2. Research and gather information on the top 3 trends in agriculture for 2025
  3. Analyze and categorize the gathered information into relevant trends
  4. Evaluate and prioritize the top 3 trends based on relevance and impact
  5. Create a concise summary of the top 3 trends in agriculture for 2025


{'goal': 'Research and summarise the top 3 trends in agriculture for 2025',
 'tasks': ['Identify credible sources of agricultural research and trends',
  'Research and gather information on the top 3 trends in agriculture for 2025',
  'Analyze and categorize the gathered information into relevant trends',
  'Evaluate and prioritize the top 3 trends based on relevance and impact',
  'Create a concise summary of the top 3 trends in agriculture for 2025'],
 'results': [],
 'critique': '',
 'approved': False,
 'iterations': 0}

In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 35.2 MB/s eta 0:00:00


In [3]:
import fitz
from google.colab import files

uploaded = files.upload()
# pip install pymupdf
# ── Open the PDF ───────────────────────────────────
doc = fitz.open("basic-text.pdf")
print(f"Pages : {len(doc)}")
print(f"Title : {doc.metadata['title']}")
print(f"Author: {doc.metadata['author']}")
# ── Extract text from every page ───────────────────
for page_num, page in enumerate(doc, start=1):
    text = page.get_text()          # plain UTF-8 string
    print(f"\n── Page {page_num} ──")
    print(text.strip())

doc.close()


Saving basic-text.pdf to basic-text.pdf
Pages : 1
Title : Sample Document for PDF Testing
Author: 

── Page 1 ──
Sample Document for PDF Testing
Introduction
This is a simple document created to test basic PDF functionality. It includes various text formatting
options to ensure proper rendering in PDF readers.
Text Formatting Examples
1. Bold text is used for emphasis.
2. Italic text can be used for titles or subtle emphasis.
3. Strikethrough is used to show deleted text.
Lists
Here's an example of an unordered list:
Item 1
Item 2
Item 3
And here's an ordered list:
1. First item
2. Second item
3. Third item
Quote
This is an example of a block quote. It can be used to highlight important information or
citations.
Table
Header 1
Header 2
Header 3
Row 1, Col 1
Row 1, Col 2
Row 1, Col 3
Row 2, Col 1
Row 2, Col 2
Row 2, Col 3
This document demonstrates various formatting options that should translate well to PDF format.
This sample PDF file is provided by Sample-Files.com. Visit us for more